In [1]:
#Torch
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

#Torch Vision
import torchvision
from torchvision import transforms
from torchvision.transforms import ToTensor
from torchvision import datasets

# Optimizers and loss function
from torch.optim import SGD
from torch.nn import CrossEntropyLoss

#Plotting
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit

In [2]:
!nvidia-smi

Wed Feb 11 21:28:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.80                 Driver Version: 581.80         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   63C    P0             16W /   85W |       0MiB /   6141MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
device

'cuda'

In [5]:
data_dir = "Data/1/PCB_DATASET/images"

data_transforms = transforms.Compose([
    transforms.Resize((224,224)),
])

pcb_dataset = datasets.ImageFolder(root=data_dir,
                                   transform=data_transforms)


# Check the classes found
print(f"Detected classes: {pcb_dataset.classes}")

Detected classes: ['Missing_hole', 'Mouse_bite', 'Open_circuit', 'Short', 'Spur', 'Spurious_copper']


In [6]:
len(pcb_dataset)

693

In [7]:
#Image 1

pcb_dataset[0]

(<PIL.Image.Image image mode=RGB size=224x224>, 0)

In [8]:
image, label = pcb_dataset[0]    
label

0

In [9]:
# Matplotlib expects (H,W,C) 

# plt.imshow(image.permute(1,2,0))
# plt.axis(False)

In [10]:
# # seeing image with only one color channel

# plt.imshow(image[0])
# plt.axis(False)

In [ ]:
labels = pcb_dataset.targets
# labels

In [12]:
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, test_idx = next(
    sss.split(range(len(pcb_dataset)), labels)
)


In [13]:
len(train_idx), len(test_idx)

(554, 139)

In [14]:
from torch.utils.data import Subset

train_dataset = Subset(pcb_dataset, train_idx)
test_dataset = Subset(pcb_dataset, test_idx)

In [15]:
len(train_dataset), len(test_dataset)

(554, 139)

In [16]:
class CustomSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        # Get original image and label from subset
        image, label = self.subset[idx]
        
        # Apply transform dynamically
        if self.transform:
            image = self.transform(image)
        
        return image, label

In [17]:
train_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomRotation(15),
    transforms.GaussianBlur(3),
    transforms.ToTensor(),
    ]
)

test_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
    ]
)

In [18]:
train_dataset = CustomSubset(train_dataset, transform=train_transforms)
test_dataset  = CustomSubset(test_dataset, transform=test_transforms)


In [19]:
print(train_dataset)

In [20]:
!nvidia-smi

Wed Feb 11 21:28:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.80                 Driver Version: 581.80         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   62C    P0             12W /   85W |       0MiB /   6141MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Building a Data Loader

In [21]:
BATCH_SIZE = 16

In [22]:
train_dataloader = DataLoader(dataset=train_dataset,
                              batch_size=BATCH_SIZE,
                              shuffle=True,
                              # num_workers=8
                            )

test_dataloader = DataLoader(dataset=test_dataset,
                              batch_size=BATCH_SIZE,
                              shuffle=False,
                              # num_workers=8
                              )


In [23]:
train_dataloader, test_dataloader

(<torch.utils.data.dataloader.DataLoader at 0x2959d3f1f30>,
 <torch.utils.data.dataloader.DataLoader at 0x295d6028d00>)

In [24]:
print(f"Length of Train Dataloader: {len(train_dataloader)} and shape: {train_dataloader.batch_size}")
print(f"Length of Test Dataloader: {len(test_dataloader)} and shape: {test_dataloader.batch_size}")

Length of Train Dataloader: 35 and shape: 16
Length of Test Dataloader: 9 and shape: 16


In [25]:
!nvidia-smi

Wed Feb 11 21:28:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.80                 Driver Version: 581.80         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   62C    P0             11W /   86W |       0MiB /   6141MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [26]:
train_features_batch, train_labels_batch = next(iter(train_dataloader))



In [27]:
!nvidia-smi

Wed Feb 11 21:28:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.80                 Driver Version: 581.80         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   63C    P0             16W /   89W |       0MiB /   6141MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [28]:
class_name = pcb_dataset.classes
class_name

['Missing_hole',
 'Mouse_bite',
 'Open_circuit',
 'Short',
 'Spur',
 'Spurious_copper']

In [29]:
# img , label = train_features_batch[0], train_labels_batch[0]

# plt.imshow(img.permute(1,2,0))
# plt.axis(False)
# plt.title(class_name[label])


In [30]:
train_labels_batch[0]

tensor(2)

In [31]:
pcb_dataset.class_to_idx

{'Missing_hole': 0,
 'Mouse_bite': 1,
 'Open_circuit': 2,
 'Short': 3,
 'Spur': 4,
 'Spurious_copper': 5}

In [32]:
train_features_batch.shape

torch.Size([16, 3, 224, 224])

## Creating a Baseline Model without any Non-linearity

In [65]:
class LinearBaseline(nn.Module):
    def __init__(self, in_features, hidden_units, out_features) -> None:
        super().__init__()

        self.layer_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=in_features,
                      out_features=hidden_units),
            nn.Linear(in_features=hidden_units,
                      out_features=out_features)
        )

    
    def forward(self, X):
        return self.layer_stack(X)
# class LinearBaseline(nn.Module):
#     def __init__(self, in_features, hidden_units, out_features) -> None:
#         super().__init__()

#         self.layer1 = nn.Flatten()
#         self.layer2 = nn.Linear(in_features=in_features,
#                       out_features=hidden_units)
#         self.layer3 = nn.Linear(in_features=hidden_units,
#                       out_features=out_features)
    
#     def forward(self, X):
#         x = self.layer1(X)
#         print(f"Forward 1: {x.shape}")
#         x = self.layer2(X)
#         print(f"Forward 2: {x.shape}")
#         x = self.layer3(X)

In [34]:
model_0 = LinearBaseline(in_features=3*224*224,
                            hidden_units=50,
                            out_features=len(pcb_dataset.classes)).to(device=device)

model_0

LinearBaseline(
  (layer_stack): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=150528, out_features=50, bias=True)
    (2): Linear(in_features=50, out_features=6, bias=True)
  )
)

In [35]:
loss_fn = CrossEntropyLoss()
optimizer = SGD(params=model_0.parameters(),
                lr=0.01)

In [36]:
from timeit import default_timer as timer

def print_train_time(start, end, device):
    total_time = end-start
    print(f"Train time on {device}: {total_time:.2f}")

In [37]:
from tqdm.auto import tqdm

In [38]:
epochs = 3
loss = 0

train_time_on_gpu = timer()
for Epoch in tqdm(range(epochs)):
    print(f"Epoch: {Epoch} \n-----------")
    train_loss = 0
    
    for batch, (X ,y) in enumerate(train_dataloader):
        model_0.train()

        X = X.to(device)
        y = y.to(device)

        y_pred = model_0(X)

        loss += loss_fn(y_pred, y)

        optimizer.zero_grad()

        

  0%|          | 0/3 [00:00<?, ?it/s]

Epoch: 0 
-----------
Epoch: 1 
-----------
Epoch: 2 
-----------


In [39]:
!nvidia-smi

Wed Feb 11 20:43:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.80                 Driver Version: 581.80         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   54C    P8              1W /   94W |    1195MiB /   6141MiB |     58%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Model with Non-Linearity 

In [40]:
class NonLinearBaseline(nn.Module):
    def __init__(self, in_features, hidden_units, out_features) -> None:
        super().__init__()

        self.layer_stack = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=in_features,
                      out_features=hidden_units),
            nn.ReLU(),
            nn.Linear(in_features=hidden_units,
                      out_features=hidden_units),
            nn.ReLU(),
            nn.Linear(in_features=hidden_units,
                      out_features=out_features)
        )

    
    def forward(self, X):
        return self.layer_stack(X)
# class NonLinearBaseline(nn.Module):
#     def __init__(self, in_features, hidden_units, out_features) -> None:
#         super().__init__()

#         self.layer1 =nn.Flatten()
#         self.layer2 = nn.Linear(in_features=in_features,
#                       out_features=hidden_units)
#         self.layer3 =nn.ReLU()
#         self.layer4 = nn.Linear(in_features=hidden_units,
#                       out_features=hidden_units)
#         self.layer5 =nn.ReLU()
#         self.layer6 =nn.Linear(in_features=hidden_units,
#                       out_features=out_features)


    
#     def forward(self, X):
#         x = self.layer1(X)
#         print(f"Forward 1: {x.shape}")
#         x = self.layer2(X)
#         print(f"Forward 2: {x.shape}")
#         x = self.layer3(X)
#         print(f"Forward 3: {x.shape}")
#         x = self.layer4(X)
#         print(f"Forward 4: {x.shape}")
#         x = self.layer5(X)
#         print(f"Forward 5: {x.shape}")
#         x = self.layer6(X)
#         print(f"Forward 6: {x.shape}")
        

In [41]:
model_1 = NonLinearBaseline(in_features=224*224*3,
                            hidden_units=50,
                            out_features=len(pcb_dataset.classes)).to(device=device)

model_1

NonLinearBaseline(
  (layer_stack): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=150528, out_features=50, bias=True)
    (2): ReLU()
    (3): Linear(in_features=50, out_features=50, bias=True)
    (4): ReLU()
    (5): Linear(in_features=50, out_features=6, bias=True)
  )
)

In [42]:
optimizer = SGD(params=model_1.parameters(), lr= 0.01)
loss_fn   = CrossEntropyLoss()

In [43]:
def accuracy_fn(y_pred, y_true):
    correct  = torch.eq(y_pred, y_true).sum().item() 
    accuracy = (correct/len(y_true))*100

    return accuracy

In [40]:
def train_step(dataloader,
               model,
               optimizer,
               accuracy_fn,
               loss_fn):
    
    train_loss, train_acc = 0, 0
    model.to(device)
    for batch, (X, y) in enumerate(dataloader):
        batch += 1
        # Device Agnostic
        X, y = X.to(device), y.to(device)
        #1. Forward pass
        y_pred = model(X)

        #2. Calculate the loss
        loss        = loss_fn(y_pred, y)
        train_loss += loss
        train_acc  +=  accuracy_fn(y_pred=y_pred.argmax(dim=1),
                                   y_true=y)
        
        #3. Optimizer zero grad
        optimizer.zero_grad()

        #4. Backpropagation
        loss.backward()
    
        #5. Optimizer step
        optimizer.step()
        if batch % 3 ==0 and batch!=18: # type: ignore]
            print(f"Looked at {batch* len(X)}/{len(dataloader.dataset)} samples")
        elif batch == 18:
            x = len(dataloader.dataset)/batch
            print(f"Looked at {batch* int(x)}/{len(dataloader.dataset)} samples")
        

    train_loss /= len(dataloader)
    train_acc  /= len(dataloader)


    return train_acc, train_loss

def test_step(model,
              dataloader,
              loss_fn,
              accuracy_fn):
    
    model.to(device)
    test_acc, test_loss  = 0, 0

    with torch.inference_mode():
        for X, y in dataloader:
            X, y   = X.to(device), y.to(device)
            y_pred = model(X)

            test_loss += loss_fn(y_pred, y)
            test_acc  += accuracy_fn(y_pred=y_pred.argmax(dim=1), y_true= y)
        
        test_acc  /= len(dataloader)
        test_loss /= len(dataloader)

    return test_acc, test_loss

In [45]:
torch.manual_seed(42)

# Measure time
from timeit import default_timer as timer
train_time_start = timer()

epochs = 3
for epoch in tqdm(range(epochs)):
    print(f"Epoch: {epoch}\n---------")
    train_acc, train_loss = train_step(dataloader=train_dataloader, 
                                        model=model_1, 
                                        loss_fn=loss_fn,
                                        optimizer=optimizer,
                                        accuracy_fn=accuracy_fn
                                    )
    test_acc, test_loss = test_step(dataloader=test_dataloader,
                                    model=model_1,
                                    loss_fn=loss_fn,
                                    accuracy_fn=accuracy_fn
                                )

    print(f"Train Loss: {train_loss:.3f} | Train Accuracy: {train_acc:.3f} | Test Loss: {test_loss:.3f} | Test Accuracy: {test_acc:.3f}")
train_time_end = timer()
total_train_time_model_1 = print_train_time(start=train_time_start,
                                            end=train_time_end,
                                            device=device)

  0%|          | 0/3 [00:00<?, ?it/s]

Epoch: 0
---------
Looked at 96/554 samples
Looked at 192/554 samples
Looked at 288/554 samples
Looked at 384/554 samples
Looked at 480/554 samples
Looked at 540/554 samples
Train Loss: 1.801 | Train Accuracy: 13.403 | Test Loss: 1.793 | Test Accuracy: 14.375
Epoch: 1
---------
Looked at 96/554 samples
Looked at 192/554 samples
Looked at 288/554 samples
Looked at 384/554 samples
Looked at 480/554 samples
Looked at 540/554 samples
Train Loss: 1.794 | Train Accuracy: 18.333 | Test Loss: 1.797 | Test Accuracy: 17.955
Epoch: 2
---------
Looked at 96/554 samples
Looked at 192/554 samples
Looked at 288/554 samples
Looked at 384/554 samples
Looked at 480/554 samples
Looked at 540/554 samples
Train Loss: 1.795 | Train Accuracy: 16.528 | Test Loss: 1.794 | Test Accuracy: 15.625
Train time on cuda: 185.67


## A CNN Model (TinyVGG Fork)

In [33]:
class CustomCNN(nn.Module):
    def __init__(self, input_shape, hidden_units, output_shape):
        super().__init__()

        self.block_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,
                         stride=2)
        )

        self.block_2 = nn.Sequential(
                        nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,
                         stride=2)
        )
        
        self.block_3 = nn.Sequential(
                        nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units,
                      out_channels=hidden_units,
                      kernel_size=3,
                      stride=1,
                      padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2,
                         stride=2)
        )
        
        # self.block_4 = nn.Sequential(
        #                 nn.Conv2d(in_channels=hidden_units,
        #               out_channels=hidden_units,
        #               kernel_size=3,
        #               stride=1,
        #               padding=1),
        #     nn.ReLU(),
        #     nn.Conv2d(in_channels=hidden_units,
        #               out_channels=hidden_units,
        #               kernel_size=3,
        #               stride=1,
        #               padding=1),
        #     nn.ReLU(),
        #     nn.MaxPool2d(kernel_size=2,
        #                  stride=2)
        # )

        # self.block_5 = nn.Sequential(
        #                 nn.Conv2d(in_channels=hidden_units,
        #               out_channels=hidden_units,
        #               kernel_size=3,
        #               stride=1,
        #               padding=1),
        #     nn.ReLU(),
        #     nn.Conv2d(in_channels=hidden_units,
        #               out_channels=hidden_units,
        #               kernel_size=3,
        #               stride=1,
        #               padding=1),
        #     nn.ReLU(),
        #     nn.MaxPool2d(kernel_size=2,
        #                  stride=2)
        # )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=hidden_units*7*7*16,
                      out_features=output_shape)
        )

        
    def forward(self, x):
        x = self.block_1(x)
        x = self.block_2(x)
        x = self.block_3(x)
        # x = self.block_4(x)
        # x = self.block_5(x)
        x = self.classifier(x)
        return x
        

In [34]:
model_2 = CustomCNN(input_shape=3,
                    hidden_units=10*7*7,
                    output_shape=len(class_name)).to(device)

model_2

CustomCNN(
  (block_1): Sequential(
    (0): Conv2d(3, 490, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(490, 490, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block_2): Sequential(
    (0): Conv2d(490, 490, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(490, 490, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (block_3): Sequential(
    (0): Conv2d(490, 490, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(490, 490, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_featu

In [35]:
images = torch.randn(size=(32, 3, 224, 224))
test_image = images[0]

test_image.shape

print(f"Image batch shape: {images.shape} -> [batch_size, color_channels, height, width]")
print(f"Single image shape: {test_image.shape} -> [color_channels, height, width]") 
print(f"Single image pixel values:\n{test_image}")

Image batch shape: torch.Size([32, 3, 224, 224]) -> [batch_size, color_channels, height, width]
Single image shape: torch.Size([3, 224, 224]) -> [color_channels, height, width]
Single image pixel values:
tensor([[[ 0.8754,  1.1178, -0.5163,  ...,  0.3267,  0.8940,  0.2184],
         [-0.9215, -0.1973,  0.2986,  ..., -1.0295,  0.5049, -0.0195],
         [ 0.9120,  0.5999, -0.7100,  ..., -1.4358,  1.3209, -1.4651],
         ...,
         [-0.4294,  0.0048, -0.9063,  ..., -0.2461,  2.0589,  0.4107],
         [-0.1521,  1.0706, -0.1244,  ...,  1.3586, -0.3756,  0.3912],
         [ 0.1429, -1.0443, -0.4230,  ...,  1.6006,  1.9912,  0.9301]],

        [[ 0.9193, -2.0570,  0.5312,  ..., -0.9769, -2.8251, -0.0140],
         [ 0.2208, -0.3074, -0.5072,  ...,  1.6123, -1.3857, -1.2242],
         [ 0.7059, -0.8883, -1.4686,  ..., -1.9110, -0.6117, -0.0074],
         ...,
         [ 0.9868,  0.7856, -0.0152,  ...,  0.0686, -0.0782, -0.1551],
         [-1.1294,  0.1234, -0.5128,  ...,  0.6432, -0.1

In [36]:
hidden=10

In [37]:
x = images

for i in range(5):
    i+=1
    print(f"Before Conv Layer {i}_1: {x.shape}")
    conv_layer = nn.Conv2d(in_channels=3 if(i==1) else 10,
                           out_channels=hidden,
                           kernel_size=3,
                           stride=1,
                           padding=1)
    x = conv_layer(x)
    print(f"After Conv Layer {i}_1: {x.shape}")
    
    relu_layer = nn.ReLU()
    x = relu_layer(x)
    
    print(f"Before Conv Layer {i}_2: {x.shape}")
    conv_layer = nn.Conv2d(in_channels=hidden,
                           out_channels=hidden,
                           kernel_size=3,
                           stride=1,
                           padding=1)
    x = conv_layer(x)
    print(f"After Conv Layer {i}_2: {x.shape}")
    
    relu_layer = nn.ReLU()
    x = relu_layer(x)
    
    print(f"Before Max Pool Layer {i}: {x.shape}")
    max_pool_layer = nn.MaxPool2d(kernel_size=2)
    x = max_pool_layer(x)
    print(f"After Max Pool Layer {i}: {x.shape}\n\n")

Before Conv Layer 1_1: torch.Size([32, 3, 224, 224])
After Conv Layer 1_1: torch.Size([32, 10, 224, 224])
Before Conv Layer 1_2: torch.Size([32, 10, 224, 224])
After Conv Layer 1_2: torch.Size([32, 10, 224, 224])
Before Max Pool Layer 1: torch.Size([32, 10, 224, 224])
After Max Pool Layer 1: torch.Size([32, 10, 112, 112])


Before Conv Layer 2_1: torch.Size([32, 10, 112, 112])
After Conv Layer 2_1: torch.Size([32, 10, 112, 112])
Before Conv Layer 2_2: torch.Size([32, 10, 112, 112])
After Conv Layer 2_2: torch.Size([32, 10, 112, 112])
Before Max Pool Layer 2: torch.Size([32, 10, 112, 112])
After Max Pool Layer 2: torch.Size([32, 10, 56, 56])


Before Conv Layer 3_1: torch.Size([32, 10, 56, 56])
After Conv Layer 3_1: torch.Size([32, 10, 56, 56])
Before Conv Layer 3_2: torch.Size([32, 10, 56, 56])
After Conv Layer 3_2: torch.Size([32, 10, 56, 56])
Before Max Pool Layer 3: torch.Size([32, 10, 56, 56])
After Max Pool Layer 3: torch.Size([32, 10, 28, 28])


Before Conv Layer 4_1: torch.Size(

In [38]:
loss_fn   = CrossEntropyLoss()
optimizer = SGD(params=model_2.parameters(), lr=0.01)

def accuracy_fn(y_pred, y_true):
    correct  = torch.eq(y_pred,y_true).sum().item()
    accuracy = (correct/len(y_true))*100

    return accuracy

In [ ]:
torch.manual_seed(42)

# Measure time
from timeit import default_timer as timer
from tqdm.auto import tqdm
train_time_start = timer()

epochs = 3
for epoch in tqdm(range(epochs)):
    print(f"Epoch: {epoch}\n---------")
    train_acc, train_loss = train_step(dataloader=train_dataloader, 
                                        model=model_2, 
                                        loss_fn=loss_fn,
                                        optimizer=optimizer,
                                        accuracy_fn=accuracy_fn
                                    )
    test_acc, test_loss = test_step(dataloader=test_dataloader,
                                    model=model_2,
                                    loss_fn=loss_fn,
                                    accuracy_fn=accuracy_fn
                                )

    print(f"Train Loss: {train_loss:.3f} | Train Accuracy: {train_acc:.3f} | Test Loss: {test_loss:.3f} | Test Accuracy: {test_acc:.3f}")
train_time_end = timer()
total_train_time_model_1 = print_train_time(start=train_time_start,
                                            end=train_time_end,
                                            device=device)

  0%|          | 0/3 [00:00<?, ?it/s]

Epoch: 0
---------
Looked at 48/554 samples
Looked at 96/554 samples
Looked at 144/554 samples
Looked at 192/554 samples
Looked at 240/554 samples
Looked at 540/554 samples
Looked at 336/554 samples
Looked at 384/554 samples
Looked at 432/554 samples
Looked at 480/554 samples
Looked at 528/554 samples
Train Loss: 1.797 | Train Accuracy: 12.786 | Test Loss: 1.792 | Test Accuracy: 17.929
Epoch: 1
---------
Looked at 48/554 samples
Looked at 96/554 samples
Looked at 144/554 samples
Looked at 192/554 samples
Looked at 240/554 samples
Looked at 540/554 samples
Looked at 336/554 samples
Looked at 384/554 samples
Looked at 432/554 samples
Looked at 480/554 samples
Looked at 528/554 samples
Train Loss: 1.795 | Train Accuracy: 13.393 | Test Loss: 1.792 | Test Accuracy: 16.919
Epoch: 2
---------
Looked at 48/554 samples
Looked at 96/554 samples
Looked at 144/554 samples
Looked at 192/554 samples


In [55]:
384160/24010


16.0